# DBSCAN Klaszterezés v2
## Kiegyensúlyozottabb Klaszter-eloszlással

### Főbb javítások:
- **Szélesebb epsilon tartomány tesztelése** (50-95. percentilis között)
- **Különböző MinPts értékek** (4, 10, 20)
- **Kiegyensúlyozottság metrika**: elkerüljük a "98% + apró klaszterek" helyzetet
- **Kombinált scoring**: Silhouette + Klaszter-eloszlás egyensúly
- **Exportálás: exports_dbscan_v2 mappába**

In [ ]:
# Alapvető könyvtárak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os

# Sklearn könyvtárak
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# Matplotlib beállítások
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Könyvtárak sikeresen betöltve!")

In [ ]:
# Adathalmaz beolvasása
root = r"bank+marketing\bank\bank-full.csv"
df = pd.read_csv(root, sep=';')

print("="*70)
print("ADATOK BETÖLTÉSE")
print("="*70)
print(f"Sorok száma: {len(df):,}")
print(f"Oszlopok száma: {len(df.columns)}")
print("\nElső 5 sor (age, balance):")
print(df[['age', 'balance']].head())
print("\nStatisztikák:")
print(df[['age', 'balance']].describe())

In [ ]:
# Feature Selection: age és balance
features = ['age', 'balance']
X = df[features].copy()

print("\n" + "="*70)
print("FEATURE SELECTION ÉS NORMALIZÁLÁS")
print("="*70)
print(f"Kiválasztott változók: {features}")
print(f"Adatpontok száma: {len(X):,}")
print(f"Dimenziószám: {X.shape[1]}")

# Normalizálás: StandardScaler (KRITIKUS DBSCAN-hez!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\n✓ Adatok normalizálva (StandardScaler)!")
print(f"  X_scaled alakja: {X_scaled.shape}")
print(f"  Átlag: {X_scaled.mean(axis=0)}")
print(f"  Szórás: {X_scaled.std(axis=0)}")

## ⭐ JAVÍTOTT MÓDSZERTAN: Kiegyensúlyozott Klaszter-eloszlás

### Probléma az eredeti módszertannal:
- Az ε=0.1382 (99. percentilis) **túl nagy volt**
- Eredmény: **98.6% egyetlen klaszterben** → nincs érdemi szegmentáció

### Új megközelítés:
1. **Szélesebb epsilon tartomány**: 50-99. percentilis (10 érték)
2. **Különböző MinPts értékek**: 4, 10, 20
3. **Klaszter-eloszlás egyensúly metrika**:
   - Gini index: méri a klaszter-méretek egyenlőtlenségét
   - Cél: **kerüljük a "1 óriás + sok apró klaszter" helyzetet**
4. **Kombinált scoring**:
   - 60% Silhouette Score (minőség)
   - 40% Klaszter-eloszlás egyensúly (gyakorlati használhatóság)

In [ ]:
def calculate_gini_coefficient(cluster_sizes):
    """
    Gini együttható kiszámítása a klaszter-méretek egyenlőtlenségének mérésére.
    
    Gini = 0: tökéletesen egyenlő eloszlás
    Gini = 1: maximális egyenlőtlenség (minden egy klaszterben)
    
    Kisebb Gini = jobb (egyenletesebb klaszter-méretek)
    """
    if len(cluster_sizes) == 0:
        return 1.0
    
    cluster_sizes = np.array(sorted(cluster_sizes))
    n = len(cluster_sizes)
    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * cluster_sizes)) / (n * np.sum(cluster_sizes)) - (n + 1) / n
    return gini

def calculate_balance_score(labels):
    """
    Klaszter-eloszlás egyensúly score kiszámítása (0-1, magasabb = jobb).
    
    Figyelembe veszi:
    - Gini együtthatót (egyenlőtlenség)
    - Legnagyobb klaszter arányát
    - Klaszterek számát
    """
    # Klaszter-méretek (noise nélkül)
    cluster_labels = labels[labels != -1]
    if len(cluster_labels) == 0:
        return 0.0
    
    unique_labels, counts = np.unique(cluster_labels, return_counts=True)
    
    if len(unique_labels) < 2:
        return 0.0  # Csak 1 klaszter -> rossz
    
    # Gini együttható (0-1, alacsonyabb = jobb)
    gini = calculate_gini_coefficient(counts)
    gini_score = 1 - gini  # Invertáljuk: magasabb = jobb
    
    # Legnagyobb klaszter aránya (0-1, alacsonyabb = jobb)
    largest_ratio = counts.max() / len(cluster_labels)
    largest_score = 1 - largest_ratio  # Invertáljuk
    
    # Klaszterek száma penalty (túl sok klaszter sem jó)
    n_clusters = len(unique_labels)
    if n_clusters > 10:
        cluster_penalty = 0.5  # Büntetés, ha túl sok klaszter
    elif n_clusters >= 3:
        cluster_penalty = 1.0  # Ideális tartomány: 3-10 klaszter
    else:
        cluster_penalty = 0.7  # 2 klaszter is elfogadható, de nem ideális
    
    # Kombinált score (0-1, magasabb = jobb)
    balance_score = (0.4 * gini_score + 0.4 * largest_score + 0.2 * cluster_penalty)
    
    return balance_score

print("✓ Klaszter-eloszlás egyensúly függvények definiálva!")

In [ ]:
# k-distance számítása több MinPts értékre
print("="*70)
print("K-DISTANCE GRAPH SZÁMÍTÁSA")
print("="*70)

minpts_values = [4, 10, 20]
k_distances_dict = {}

for min_pts in minpts_values:
    neighbors = NearestNeighbors(n_neighbors=min_pts)
    neighbors_fit = neighbors.fit(X_scaled)
    distances, indices = neighbors_fit.kneighbors(X_scaled)
    k_distances = distances[:, -1]
    k_distances_dict[min_pts] = k_distances
    
    print(f"\nMinPts = {min_pts}:")
    print(f"  50%: {np.percentile(k_distances, 50):.4f}")
    print(f"  75%: {np.percentile(k_distances, 75):.4f}")
    print(f"  90%: {np.percentile(k_distances, 90):.4f}")
    print(f"  95%: {np.percentile(k_distances, 95):.4f}")
    print(f"  99%: {np.percentile(k_distances, 99):.4f}")

print("\n✓ k-distance értékek kiszámítva!")

In [ ]:
# SZÉLES KÖRŰ PARAMÉTER TESZTELÉS
print("\n" + "="*100)
print("DBSCAN SZÉLES KÖRŰ PARAMÉTER TESZTELÉS")
print("="*100)

all_results = []

# Minden MinPts értékre
for min_pts in minpts_values:
    k_distances = k_distances_dict[min_pts]
    
    # Epsilon értékek: 50-99. percentilis tartományban, 10 értékkel
    percentiles = [50, 60, 70, 75, 80, 85, 90, 95, 97, 99]
    epsilon_values = [np.percentile(k_distances, p) for p in percentiles]
    
    print(f"\n{'='*100}")
    print(f"MinPts = {min_pts}")
    print(f"{'='*100}")
    
    for eps, pct in zip(epsilon_values, percentiles):
        # DBSCAN futtatása
        dbscan = DBSCAN(eps=eps, min_samples=min_pts)
        labels = dbscan.fit_predict(X_scaled)
        
        # Statisztikák
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        noise_ratio = n_noise / len(labels) * 100
        n_core = len(dbscan.core_sample_indices_)
        
        # Klaszter-eloszlás elemzés
        cluster_labels = labels[labels != -1]
        if len(cluster_labels) > 0:
            unique_labels, counts = np.unique(cluster_labels, return_counts=True)
            largest_cluster_pct = counts.max() / len(cluster_labels) * 100 if len(counts) > 0 else 0
        else:
            largest_cluster_pct = 0
        
        # Validációs metrikák (noise nélkül)
        if n_clusters >= 2 and noise_ratio < 50:
            mask = labels != -1
            if mask.sum() > 1:
                try:
                    silhouette = silhouette_score(X_scaled[mask], labels[mask])
                    davies_bouldin = davies_bouldin_score(X_scaled[mask], labels[mask])
                    calinski = calinski_harabasz_score(X_scaled[mask], labels[mask])
                except:
                    silhouette = davies_bouldin = calinski = np.nan
            else:
                silhouette = davies_bouldin = calinski = np.nan
        else:
            silhouette = davies_bouldin = calinski = np.nan
        
        # Klaszter-eloszlás egyensúly score
        balance_score = calculate_balance_score(labels)
        
        # Kombinált score: 60% Silhouette + 40% Balance
        if not np.isnan(silhouette) and balance_score > 0:
            # Normalizálás: Silhouette -1..1 -> 0..1
            silhouette_normalized = (silhouette + 1) / 2
            combined_score = 0.6 * silhouette_normalized + 0.4 * balance_score
        else:
            combined_score = 0
        
        all_results.append({
            'min_pts': min_pts,
            'epsilon': eps,
            'percentile': pct,
            'n_clusters': n_clusters,
            'n_core': n_core,
            'n_noise': n_noise,
            'noise_ratio': noise_ratio,
            'largest_cluster_pct': largest_cluster_pct,
            'silhouette': silhouette,
            'davies_bouldin': davies_bouldin,
            'calinski_harabasz': calinski,
            'balance_score': balance_score,
            'combined_score': combined_score,
            'labels': labels,
            'dbscan': dbscan
        })
        
        # Rövid kimenet
        status = "✓" if noise_ratio <= 15 and largest_cluster_pct < 80 else "⚠"
        print(f"ε={eps:.4f} ({pct}%): Klaszter={n_clusters:3d}, Noise={noise_ratio:5.1f}%, "
              f"Max={largest_cluster_pct:5.1f}%, Balance={balance_score:.3f}, "
              f"Silh={silhouette:.3f if not np.isnan(silhouette) else 'N/A':>5s}, "
              f"Combined={combined_score:.3f} {status}")

print(f"\n✓ Összesen {len(all_results)} konfiguráció tesztelve!")

In [ ]:
# OPTIMÁLIS KONFIGURÁCIÓ KIVÁLASZTÁSA
print("\n" + "="*100)
print("OPTIMÁLIS KONFIGURÁCIÓ KIVÁLASZTÁSA")
print("="*100)

# Szűrés: noise <= 20% ÉS legnagyobb klaszter < 85% ÉS legalább 2 klaszter
valid_results = [
    r for r in all_results 
    if r['noise_ratio'] <= 20 
    and r['largest_cluster_pct'] < 85
    and r['n_clusters'] >= 2
    and not np.isnan(r['silhouette'])
]

print(f"\nSzűrési kritériumok:")
print(f"  - Noise arány ≤ 20%")
print(f"  - Legnagyobb klaszter < 85%")
print(f"  - Legalább 2 klaszter")
print(f"  - Silhouette Score létezik")
print(f"\nMegfelelő konfigurációk száma: {len(valid_results)}")

if len(valid_results) > 0:
    # TOP 5 legjobb kombinált score
    top5 = sorted(valid_results, key=lambda x: x['combined_score'], reverse=True)[:5]
    
    print("\n" + "="*100)
    print("TOP 5 LEGJOBB KONFIGURÁCIÓ (kombinált score alapján)")
    print("="*100)
    print(f"{'Rang':<5} {'MinPts':<8} {'ε':<10} {'Klaszter':<10} {'Noise%':<8} {'MaxKlasz%':<10} "
          f"{'Silh':<8} {'Balance':<9} {'Combined':<9}")
    print("-" * 100)
    
    for i, r in enumerate(top5, 1):
        print(f"{i:<5} {r['min_pts']:<8} {r['epsilon']:<10.4f} {r['n_clusters']:<10} "
              f"{r['noise_ratio']:<8.2f} {r['largest_cluster_pct']:<10.1f} "
              f"{r['silhouette']:<8.4f} {r['balance_score']:<9.4f} {r['combined_score']:<9.4f}")
    
    # Legjobb kiválasztása
    optimal_result = top5[0]
    
    print("\n" + "="*100)
    print("⭐ OPTIMÁLIS KONFIGURÁCIÓ KIVÁLASZTVA ⭐")
    print("="*100)
    print(f"MinPts:                 {optimal_result['min_pts']}")
    print(f"Epsilon (ε):            {optimal_result['epsilon']:.4f} ({optimal_result['percentile']}. percentilis)")
    print(f"Klaszterek száma:       {optimal_result['n_clusters']}")
    print(f"Noise arány:            {optimal_result['noise_ratio']:.2f}% ✓")
    print(f"Legnagyobb klaszter:    {optimal_result['largest_cluster_pct']:.1f}% ✓")
    print(f"Silhouette Score:       {optimal_result['silhouette']:.4f}")
    print(f"Davies-Bouldin Index:   {optimal_result['davies_bouldin']:.4f}")
    print(f"Calinski-Harabasz:      {optimal_result['calinski_harabasz']:.2f}")
    print(f"Balance Score:          {optimal_result['balance_score']:.4f}")
    print(f"Combined Score:         {optimal_result['combined_score']:.4f}")
    print("="*100)
    
else:
    print("\n⚠ FIGYELMEZTETÉS: Nincs olyan konfiguráció, amely megfelel a kritériumoknak!")
    print("\nLegalacsonyabb noise arányú konfiguráció választása...")
    optimal_result = min(all_results, key=lambda x: x['noise_ratio'])
    print(f"\nKiválasztva: MinPts={optimal_result['min_pts']}, ε={optimal_result['epsilon']:.4f}")
    print(f"  Noise: {optimal_result['noise_ratio']:.2f}%")
    print(f"  Klaszterek: {optimal_result['n_clusters']}")

# Címkék hozzáadása az adathalmazhoz
optimal_labels = optimal_result['labels']
optimal_dbscan = optimal_result['dbscan']
optimal_eps = optimal_result['epsilon']
optimal_min_pts = optimal_result['min_pts']

df['cluster'] = optimal_labels

In [ ]:
# KLASZTER PROFILOK RÉSZLETES ELEMZÉSE
print("\n" + "="*100)
print("KLASZTER PROFILOK RÉSZLETES ELEMZÉSE")
print("="*100)

profile_data = []

for cluster_id in sorted(df['cluster'].unique()):
    cluster_df = df[df['cluster'] == cluster_id]
    
    profile_data.append({
        'Klaszter': 'Noise' if cluster_id == -1 else f'Cluster_{cluster_id}',
        'Klaszter_ID': cluster_id,
        'Elemszám (db)': len(cluster_df),
        'Elemszám (%)': len(cluster_df) / len(df) * 100,
        'Átlag age': cluster_df['age'].mean(),
        'Átlag balance': cluster_df['balance'].mean(),
        'Min age': cluster_df['age'].min(),
        'Max age': cluster_df['age'].max(),
        'Min balance': cluster_df['balance'].min(),
        'Max balance': cluster_df['balance'].max()
    })

profile_df = pd.DataFrame(profile_data)
profile_df = profile_df.sort_values('Elemszám (db)', ascending=False).reset_index(drop=True)

print("\nKlaszter profilok:")
print(profile_df.to_string(index=False))

# CSV exportálás
profile_df.to_csv('exports_dbscan_v2/dbscan_v2_cluster_profiles.csv', index=False)
print("\n✓ Klaszter profilok exportálva: exports_dbscan_v2/dbscan_v2_cluster_profiles.csv")

In [ ]:
# SCATTER PLOT VIZUALIZÁCIÓ
fig, ax = plt.subplots(figsize=(14, 10))

# Klaszterek szétválasztása noise-tól
mask_clusters = df['cluster'] != -1
mask_noise = df['cluster'] == -1

# Noise pontok (szürke, kis méret)
if mask_noise.sum() > 0:
    ax.scatter(df[mask_noise]['age'], df[mask_noise]['balance'], 
              c='lightgray', s=20, alpha=0.4, label='Noise', edgecolors='none')

# Klaszterek (színes)
if mask_clusters.sum() > 0:
    scatter = ax.scatter(df[mask_clusters]['age'], df[mask_clusters]['balance'], 
                        c=df[mask_clusters]['cluster'], cmap='tab20', 
                        s=50, alpha=0.7, edgecolors='black', linewidths=0.5)
    plt.colorbar(scatter, ax=ax, label='Klaszter ID')

ax.set_xlabel('Életkor (age)', fontsize=12, fontweight='bold')
ax.set_ylabel('Egyenleg (balance) €', fontsize=12, fontweight='bold')
ax.set_title(f'DBSCAN Klaszterezés Eredményei (v2)\n' +
            f'MinPts={optimal_min_pts}, ε={optimal_eps:.4f}, ' +
            f'Klaszterek={optimal_result["n_clusters"]}, ' +
            f'Noise={optimal_result["noise_ratio"]:.1f}%',
            fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('exports_dbscan_v2/dbscan_v2_scatter_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Scatter plot mentve: exports_dbscan_v2/dbscan_v2_scatter_plot.png")

In [ ]:
# ÖSSZEHASONLÍTÓ TÁBLÁZAT: Összes tesztelt konfiguráció
comparison_df = pd.DataFrame([{
    'MinPts': r['min_pts'],
    'Epsilon': r['epsilon'],
    'Percentilis': r['percentile'],
    'Klaszterek': r['n_clusters'],
    'Noise (%)': r['noise_ratio'],
    'Max Klaszter (%)': r['largest_cluster_pct'],
    'Silhouette': r['silhouette'],
    'Davies-Bouldin': r['davies_bouldin'],
    'Calinski-Harabasz': r['calinski_harabasz'],
    'Balance Score': r['balance_score'],
    'Combined Score': r['combined_score']
} for r in all_results])

comparison_df = comparison_df.sort_values('Combined Score', ascending=False)
comparison_df.to_csv('exports_dbscan_v2/dbscan_v2_all_configurations.csv', index=False)

print("\n✓ Összes konfiguráció exportálva: exports_dbscan_v2/dbscan_v2_all_configurations.csv")
print(f"  Összesen {len(comparison_df)} konfiguráció")

In [ ]:
# VÉGSŐ ÖSSZEFOGLALÓ
print("\n" + "="*100)
print("DBSCAN V2 - JAVÍTOTT VERZIÓ - VÉGSŐ ÖSSZEFOGLALÓ")
print("="*100)

print(f"\n1. OPTIMÁLIS PARAMÉTEREK:")
print(f"   - Epsilon (ε):         {optimal_result['epsilon']:.4f} ({optimal_result['percentile']}. percentilis)")
print(f"   - MinPts:              {optimal_result['min_pts']}")
print(f"   - Forrás:              Kombinált scoring (60% Silhouette + 40% Balance)")

print(f"\n2. DBSCAN EREDMÉNYEK:")
print(f"   - Azonosított klaszterek:  {optimal_result['n_clusters']}")
print(f"   - Core pontok:             {optimal_result['n_core']:,} ({optimal_result['n_core']/len(df)*100:.1f}%)")
print(f"   - Noise pontok:            {optimal_result['n_noise']:,} ({optimal_result['noise_ratio']:.2f}%)")
print(f"   - Legnagyobb klaszter:     {optimal_result['largest_cluster_pct']:.1f}%")

if optimal_result['noise_ratio'] <= 15:
    print(f"   ✓ Noise arány elfogadható (<= 15%)")
if optimal_result['largest_cluster_pct'] < 80:
    print(f"   ✓ Kiegyensúlyozott klaszter-eloszlás (legnagyobb < 80%)")

print(f"\n3. VALIDÁCIÓS METRIKÁK (noise nélkül):")
print(f"   - Silhouette Score:        {optimal_result['silhouette']:.4f}")
print(f"   - Davies-Bouldin Index:    {optimal_result['davies_bouldin']:.4f}")
print(f"   - Calinski-Harabasz Index: {optimal_result['calinski_harabasz']:.2f}")
print(f"   - Balance Score:           {optimal_result['balance_score']:.4f}")
print(f"   - Combined Score:          {optimal_result['combined_score']:.4f}")

print(f"\n4. KLASZTEREK (TOP 10 LEGNAGYOBB):")
top10_clusters = profile_df.head(10)
for _, row in top10_clusters.iterrows():
    label = row['Klaszter']
    count = int(row['Elemszám (db)'])
    pct = row['Elemszám (%)']
    avg_age = row['Átlag age']
    avg_balance = row['Átlag balance']
    
    if label == 'Noise':
        print(f"   {label}: {count:>6,} pont ({pct:>5.1f}%) - Outliers")
    else:
        print(f"   {label}: {count:>6,} pont ({pct:>5.1f}%) - "
              f"Átlag kor: {avg_age:.0f} év, Átlag egyenleg: {avg_balance:,.0f} €")

print(f"\n5. EXPORTÁLT FÁJLOK:")
print(f"   - exports_dbscan_v2/dbscan_v2_cluster_profiles.csv")
print(f"   - exports_dbscan_v2/dbscan_v2_all_configurations.csv")
print(f"   - exports_dbscan_v2/dbscan_v2_scatter_plot.png")

print(f"\n6. ÖSSZEHASONLÍTÁS AZ EREDETI VERZIÓVAL:")
print(f"   Eredeti (v1):")
print(f"     - MinPts=4, ε=0.1382 (99. percentilis)")
print(f"     - 36 klaszter, legnagyobb klaszter: 98.6% ⚠")
print(f"     - Probléma: Nincs érdemi szegmentáció")
print(f"\n   Javított (v2):")
print(f"     - MinPts={optimal_result['min_pts']}, ε={optimal_result['epsilon']:.4f} ({optimal_result['percentile']}. percentilis)")
print(f"     - {optimal_result['n_clusters']} klaszter, legnagyobb klaszter: {optimal_result['largest_cluster_pct']:.1f}%")
if optimal_result['largest_cluster_pct'] < 80:
    print(f"     - ✓ Kiegyensúlyozottabb klaszter-eloszlás!")

print("\n" + "="*100)
print("✓ DBSCAN V2 KLASZTEREZÉS SIKERESEN BEFEJEZVE!")
print("="*100)